# TALLER 3 — CONCURRENCIA
**Curso:** Infraestructuras Paralelas y Distribuidas — 750023C

**Profesor:** Manuel Alejandro Pastrana, PhD

**Estudiante:** Juan Fernando Duque Jojoa - 2610342

Taller:
1. Un problema del mundo real resuelto con el enfoque de la sesión
2. Un programa concurrente, **sin** reutilizar los ejemplos de clase.
3. Explicación detallada del código + análisis de fortalezas y debilidades.

Ejemplos de la clase: Contar palabras, Suma de cuadrados, Demos concurrencia

# 1. Identificación del Problema del Mundo Real

## Contexto
En la administracion de infraestructuras web y servicios en la nube, los servidores generan de forma continua registros de auditoria y trafico en formato textual estandar, comunmente son los formatos log's. En plataformas de alto trafico, estos archivos pueden acumular cientos, miles o hasta millones de lineas de peticiones HTTP en poco tiempo

Se debe analizar estos procesos para temas de ciberseguridad, o incluso para saber si la plataforma que esta al aire, realmente siga funcionando y no mande errores. Metricas clave que podemos usar con los Log's:

1. **Distribución de códigos de estado HTTP:** Conocer el balance entre respuestas exitosas (200), redirecciones (301, 302), errores de cliente (403, 404) y fallos internos de servidor (500, 502).

2. **Consumo total de ancho de banda:** Acumular el total de bytes transferidos a clientes externos para dimensionamiento de red y costos de egreso.

3. **Detección de anomalías y clientes recurrentes:** Identificar direcciones IP sospechosas que generen tasas desproporcionadas de solicitudes (posibles ataques de denegación de servicio o scrapers no autorizados).

### Enfoque de Solución Concurrente
El procesamiento secuencial línea por línea resulta ineficiente por la cantidad de datos que pasan en poco tiempo. Aplicando los fundamentos de concurrencia:

   Se particiona el conjunto de logs en segmentos, llamados chunks, asignados a multiples hilos de trabajos independientes

   Cada hilo analiza las lineas de su particion y acumula sus estadisticas de forma privada y desacoplada (para evitar la contencion constante sobre la memoria compartida)

   Se consolida la informacion global mediante una fase de reduccion


## 2. Generación del Dataset Sintético de Prueba
Para garantizar la reproducibilidad y pruebas sin dependencias externas de archivos, generamos un conjunto representativo de 120,000 líneas de logs en memoria simulando accesos HTTP reales.

PD: Propuesta de codigo de Dataset Random propuesta por IA

In [ ]:
import random
import time
import threading
from collections import defaultdict

# Configuración del dataset
NUM_LINEAS = 120_000 # Número de líneas en el dataset
IPS_FRECUENTES = [f"192.168.1.{i}" for i in range(1, 25)] + [f"10.0.0.{i}" for i in range(1, 10)]
METODOS = ["GET", "POST", "PUT", "DELETE"]
RUTAS = ["/index.html", "/api/v1/login", "/dashboard", "/static/bundle.js", "/api/v1/data", "/images/logo.png"]
CODIGOS_HTTP = [200, 200, 200, 200, 301, 403, 404, 404, 500, 502]

print(f"Generando dataset de {NUM_LINEAS:,} líneas de log...")
random.seed(42)
logs = []
for _ in range(NUM_LINEAS):
    ip = random.choice(IPS_FRECUENTES)
    metodo = random.choice(METODOS)
    ruta = random.choice(RUTAS)
    codigo = random.choice(CODIGOS_HTTP)
    bytes_enviados = random.randint(150, 15000) if codigo == 200 else random.randint(40, 500)
    linea = f'{ip} - - [01/Oct/2026:12:00:00 -0500] "{metodo} {ruta} HTTP/1.1" {codigo} {bytes_enviados}'
    logs.append(linea)

print(f"Dataset generado con éxito. Ejemplo de línea:\n{logs[0]}")

Generando dataset de 120,000 líneas de log...
Dataset generado con éxito. Ejemplo de línea:
192.168.1.8 - - [01/Oct/2026:12:00:00 -0500] "GET /images/logo.png HTTP/1.1" 301 165


## 3. Implementación Secuencial (Línea Base)
Implementamos la versión tradicional en un único hilo para tener una referencia exacta de correctitud métrica y tiempo de ejecución.

In [4]:
def procesar_logs_secuencial(lineas_log):
    conteo_estados = defaultdict(int)
    total_bytes = 0
    peticiones_ip = defaultdict(int)
    
    for linea in lineas_log:
        partes = linea.split()
        ip = partes[0]
        codigo = int(partes[-2])
        tamano = int(partes[-1])
        
        conteo_estados[codigo] += 1
        total_bytes += tamano
        peticiones_ip[ip] += 1
        
    return conteo_estados, total_bytes, peticiones_ip

t_inicio_sec = time.perf_counter()
estados_sec, bytes_sec, ips_sec = procesar_logs_secuencial(logs)
t_fin_sec = time.perf_counter()
tiempo_secuencial = t_fin_sec - t_inicio_sec

print(f"[Secuencial] Completado en {tiempo_secuencial:.4f} s")
print(f"Total Bytes: {bytes_sec:,}")
print(f"Distribución Códigos: {dict(estados_sec)}")
top_ip_sec = max(ips_sec.items(), key=lambda x: x[1])
print(f"IP con mayor tráfico: {top_ip_sec[0]} ({top_ip_sec[1]} solicitudes)")

[Secuencial] Completado en 0.0791 s
Total Bytes: 380,446,302
Distribución Códigos: {301: 12014, 200: 47780, 502: 12022, 404: 23971, 403: 12131, 500: 12082}
IP con mayor tráfico: 192.168.1.6 (3752 solicitudes)


## 4. Implementación Concurrente (Hilos con Sección Crítica Mínima)

### Arquitectura de la Solución:
1. **División de datos:** El array general de logs lo distribuiremos en $N$ bloques balanceados.
2. **Aislamiento de cómputo:** Cada hilo ejecuta la función `hilo_analizador`, procesando de manera privada su partición en variables locales
3. **Sincronización:** Una vez completado su bloque, el hilo adquiere el lock global `lock_global` durante una sección crítica mínima para consolidar sus totales hacia las estructuras compartidas.

In [ ]:
NUM_HILOS = 4

# Estructuras compartidas globales para la consolidación
global_estados = defaultdict(int)
global_bytes = 0
global_ips = defaultdict(int)
lock_global = threading.Lock()

# Función que cada hilo ejecutará para procesar su partición de logs
def hilo_analizador(id_hilo, particion_logs):
    global global_bytes
    
    # Variables privadas y locales por hilo (sin locks, ejecución paralela)
    local_estados = defaultdict(int)
    local_bytes = 0
    local_ips = defaultdict(int)
    
    for linea in particion_logs:
        partes = linea.split()
        ip = partes[0]
        codigo = int(partes[-2])
        tamano = int(partes[-1])
        
        local_estados[codigo] += 1
        local_bytes += tamano
        local_ips[ip] += 1
        
    # Sección Crítica Mínima: Solo al consolidar los resultados acumulados
    with lock_global:
        global_bytes += local_bytes
        for codigo, cant in local_estados.items():
            global_estados[codigo] += cant
        for ip, cant in local_ips.items():
            global_ips[ip] += cant

# Función para procesar logs de manera concurrente
def procesar_logs_concurrente(lineas_log, n_hilos=4):
    global global_estados, global_bytes, global_ips
    global_estados = defaultdict(int)
    global_bytes = 0
    global_ips = defaultdict(int)
    
    tam_chunk = len(lineas_log) // n_hilos
    hilos = []
    
    t_inicio = time.perf_counter()

    # Crear y lanzar hilos
    for i in range(n_hilos):
        inicio = i * tam_chunk
        fin = inicio + tam_chunk if i < n_hilos - 1 else len(lineas_log)
        particion = lineas_log[inicio:fin]
        
        t = threading.Thread(target=hilo_analizador, args=(i, particion))
        hilos.append(t)
        t.start()

    # Esperar a que todos los hilos terminen 
    for t in hilos:
        t.join()
        
    t_fin = time.perf_counter()
    return t_fin - t_inicio

tiempo_concurrente = procesar_logs_concurrente(logs, NUM_HILOS)

print(f"[Concurrente con {NUM_HILOS} hilos] Completado en {tiempo_concurrente:.4f} s")
print(f"Total Bytes: {global_bytes:,}")
print(f"Distribución Códigos: {dict(global_estados)}")
top_ip_conc = max(global_ips.items(), key=lambda x: x[1])
print(f"IP con mayor tráfico: {top_ip_conc[0]} ({top_ip_conc[1]} solicitudes)")

[Concurrente con 4 hilos] Completado en 0.1115 s
Total Bytes: 380,446,302
Distribución Códigos: {301: 12014, 200: 47780, 502: 12022, 404: 23971, 403: 12131, 500: 12082}
IP con mayor tráfico: 192.168.1.6 (3752 solicitudes)
